In Chapter 4, we learned that attention lets a Query look at a set of Keys and retrieve information from their corresponding Values.

Now we apply that idea to a sequence of tokens.

Consider:

"The cat sat on the mat because it was tired."

When the model processes "it", it needs to understand what "it" refers to.

Self-attention allows every token to look at the other tokens in the same sequence and determine which ones are relevant.

So instead of:

Query → external Keys/Values

we have:

Tokens → Queries, Keys, and Values

Every token produces its own Q, K, V.

That is why it is called self-attention.

In [1]:
import numpy as np

X=np.array([
    [1.0,0.0],
    [0.0,1.0],
    [1.0,1.0],
    [0.5,1.0]
])

print("Shape:",X.shape)
print(X)

Shape: (4, 2)
[[1.  0. ]
 [0.  1. ]
 [1.  1. ]
 [0.5 1. ]]


In [2]:
W_Q=np.array([
    [1.0,0.0],
    [0.0,1.0]
])

W_K=np.array([
    [1.0,0.0],
    [0.0,1.0]
])
W_V=np.array([
    [1.0,0.0],
    [0.0,1.0]
])


Q=X@W_Q
K=X@W_K
V=X@W_V

print("Q:\n", Q)
print("\nK:\n", K)
print("\nV:\n", V)

Q:
 [[1.  0. ]
 [0.  1. ]
 [1.  1. ]
 [0.5 1. ]]

K:
 [[1.  0. ]
 [0.  1. ]
 [1.  1. ]
 [0.5 1. ]]

V:
 [[1.  0. ]
 [0.  1. ]
 [1.  1. ]
 [0.5 1. ]]


Self-Attention Scores

Now comes the important part.

Every token compares its Query with every Key.

We calculate:

$$ Scores = QK^T $$

If we have 4 tokens:

$$ Q \in \mathbb{R}^{4\times2} $$

and

$$ K^T \in \mathbb{R}^{2\times4} $$

therefore:

$$ QK^T \in \mathbb{R}^{4\times4} $$

So we get a 4 × 4 attention score matrix.

Each row represents one token asking:

"How relevant is every other token to me?"

In [4]:
scores=Q@K.T
print("Score matrix shape:",scores.shape)
print(scores)

Score matrix shape: (4, 4)
[[1.   0.   1.   0.5 ]
 [0.   1.   1.   1.  ]
 [1.   1.   2.   1.5 ]
 [0.5  1.   1.5  1.25]]


             Keys
          1   2   3   4
       ┌───────────────
Query 1│
      2│
      3│
      4│

Scaling
📝 Markdown Cell

For large vectors, dot products can become large.

Large scores can make softmax extremely sharp.

Therefore Transformers divide the scores by:

$$ \sqrt{d_k} $$

where \(d_k\) is the dimensionality of the Key vectors.

The scaled scores are:

$$ S=\frac{QK^T}{\sqrt{d_k}} $$

For our example:

$$ d_k=2 $$

so:

d
k
	​

	​

=
2
	​


In [6]:
d_k=K.shape[1]

scaled_scores=scores/np.sqrt(d_k)

print("d_k:",d_k)
print("Scaled scores:\n",scaled_scores)

d_k: 2
Scaled scores:
 [[0.70710678 0.         0.70710678 0.35355339]
 [0.         0.70710678 0.70710678 0.70710678]
 [0.70710678 0.70710678 1.41421356 1.06066017]
 [0.35355339 0.70710678 1.06066017 0.88388348]]


In [7]:
def softmax_rows(x):
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

weights = softmax_rows(scaled_scores)

print("Attention weights:\n", weights)
print("\nRow sums:", weights.sum(axis=1))

Attention weights:
 [[0.31296385 0.15431268 0.31296385 0.21975962]
 [0.14115631 0.28628123 0.28628123 0.28628123]
 [0.18341106 0.18341106 0.37197871 0.26119917]
 [0.16255597 0.23149905 0.3296822  0.27626277]]

Row sums: [1. 1. 1. 1.]


In [8]:
output = weights @ V

print("Output shape:", output.shape)
print(output)

Output shape: (4, 2)
[[0.73580751 0.68703615]
 [0.57057816 0.85884369]
 [0.68598936 0.81658894]
 [0.63036956 0.83744403]]
